# Fine-tuning Mask R-CNN — Détection de Fissures

Ce notebook exécute l'entraînement complet de Mask R-CNN (via Detectron2) pour la segmentation d'instances de fissures.

## 1. Montage de Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration des chemins
Chaque dataset est une variable de chemin **indépendante** : collez le chemin réel
de chaque dossier partagé de votre Drive (peu importe où il se trouve).

In [ ]:
import os, shutil

# ─── À adapter ────────────────────────────────────────────
DRIVE_PROJECT = '/content/drive/MyDrive/Projet_Fissures'

# Dataset COCO (pour Mask R-CNN) : dossier avec train/valid/test + _annotations.coco.json
DATASET_COCO = '/content/drive/MyDrive/segmentation_fissures.v7i.coco-segmentation'

# Murs sains (optionnel) : dossier d'images sans fissure
MURS_SAINS = '/content/drive/MyDrive/murs_sains'

# Si le raccourci ci-dessus est "cassé" (ls renvoie No such file or directory
# alors que le dossier apparaît dans MyDrive avec des droits 'l...'), utiliser
# plutôt les chemins directs par ID de raccourci (voir README, section
# "raccourci Drive cassé") :
# DATASET_COCO = '/content/drive/.shortcut-targets-by-id/1oMkn8F0pVPcXN83ThSL5mMlAXWWsqf_W/segmentation_fissures.v7i.coco-segmentation'
# MURS_SAINS   = '/content/drive/.shortcut-targets-by-id/1Kan2sZwrdw_F-ZrYm7_tAwxBfE6s2hiL/murs_sains'
# ──────────────────────────────────────────────────────────

PROJECT_DIR = '/content/Projet'

shutil.copytree(DRIVE_PROJECT, PROJECT_DIR, dirs_exist_ok=True)
os.chdir(PROJECT_DIR)

for path, label in [(DATASET_COCO, 'Dataset COCO'), (MURS_SAINS, 'Murs sains')]:
    if os.path.isdir(path):
        print(f'  [OK] {label} : {path}  ->  {os.listdir(path)[:5]}')
    else:
        print(f'  [INTROUVABLE] {label} : {path}')
print(f'Prêt — projet : {PROJECT_DIR}')

## 3. Installation de Detectron2

In [ ]:
import torch
print(f'PyTorch : {torch.__version__} | CUDA : {torch.version.cuda}')

# Installation automatique selon la version CUDA
cuda_ver = torch.version.cuda.replace('.', '')[:3]  # ex: '118'
torch_ver = torch.__version__.split('+')[0].replace('.', '')[:3]  # ex: '210'

!pip install -q opencv-python-headless pycocotools pyyaml tqdm
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
print('Detectron2 installé.')

## 4. Vérification du GPU

In [ ]:
import torch
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 5. Entraînement Mask R-CNN

In [ ]:
!python scripts/train_maskrcnn.py \
    --data-root {DATASET_COCO} \
    --murs-sains {MURS_SAINS} \
    --config configs/maskrcnn_config.yaml

In [ ]:
# ─── Surcharges optionnelles ───
# !python scripts/train_maskrcnn.py \
#     --data-root {DATASET_COCO} \
#     --murs-sains {MURS_SAINS} \
#     --config configs/maskrcnn_config.yaml \
#     --max-iter 15000 \
#     --batch 2 \
#     --lr 0.00005 \
#     --output-dir outputs/maskrcnn/run2

## 6. Reprendre un entraînement interrompu

In [ ]:
# Reprendre depuis le dernier checkpoint automatique
# !python scripts/train_maskrcnn.py \
#     --data-root {DATASET_COCO} \
#     --resume

# Ou depuis un checkpoint spécifique
# !python scripts/train_maskrcnn.py \
#     --data-root {DATASET_COCO} \
#     --resume-from outputs/maskrcnn/run/model_0005000.pth

## 7. Visualisation des métriques

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

metrics_file = Path('outputs/maskrcnn/run/metrics.json')
if metrics_file.exists():
    records = []
    with open(metrics_file) as f:
        for line in f:
            try:
                records.append(json.loads(line.strip()))
            except json.JSONDecodeError:
                pass

    iters  = [r.get('iteration', i) for i, r in enumerate(records)]
    losses = [r.get('total_loss') for r in records]
    map50  = [r.get('bbox/AP50') for r in records]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    valid_loss = [(it, l) for it, l in zip(iters, losses) if l is not None]
    if valid_loss:
        ax1.plot(*zip(*valid_loss), label='Total loss')
        ax1.set_title('Perte totale')
        ax1.set_xlabel('Itération')
        ax1.legend()

    valid_map = [(it, m) for it, m in zip(iters, map50) if m is not None]
    if valid_map:
        ax2.plot(*zip(*valid_map), color='green', label='BBox AP50')
        ax2.set_title('mAP@50 (BBox)')
        ax2.set_xlabel('Itération')
        ax2.legend()

    plt.tight_layout()
    plt.show()
else:
    print(f'Fichier metrics.json introuvable : {metrics_file}')

## 8. Sauvegarde vers Drive

In [ ]:
import shutil
dest = f'{DRIVE_PROJECT}/outputs/maskrcnn'
shutil.copytree('outputs/maskrcnn', dest, dirs_exist_ok=True)
print(f'Résultats sauvegardés vers : {dest}')